In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = sns.load_dataset('titanic')
print(df.shape)
print(df.head())

(891, 15)
   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  


In [2]:
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
survived       891 non-null int64
pclass         891 non-null int64
sex            891 non-null object
age            714 non-null float64
sibsp          891 non-null int64
parch          891 non-null int64
fare           891 non-null float64
embarked       889 non-null object
class          891 non-null category
who            891 non-null object
adult_male     891 non-null bool
deck           203 non-null category
embark_town    889 non-null object
alive          891 non-null object
alone          891 non-null bool
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.6+ KB


survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [3]:
# age 用中位数填充
df['age'] = df['age'].fillna(df['age'].median())

# embarked 用众数填充
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

In [4]:
df['sex'] = df['sex'].map({'male': 0, 'female': 1})
df = pd.get_dummies(df, columns=['embarked'], drop_first=True)

In [5]:
# 家庭人数 = 兄弟姐妹 + 父母子女 + 自己
df['family_size'] = df['sibsp'] + df['parch'] + 1

# 是否独自一人
df['is_alone'] = (df['family_size'] == 1).astype(int)

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
            'family_size', 'is_alone', 'embarked_Q', 'embarked_S']

data = df[features + ['survived']].dropna()
X = data[features]
y = data['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("准确率:", accuracy_score(y_test, pred))

准确率: 0.7988826815642458


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    print(f"{name}: {acc:.4f}")

LogisticRegression: 0.7989
DecisionTree: 0.7989
RandomForest: 0.8268
XGBoost: 0.7989


In [8]:
from sklearn.model_selection import cross_val_score

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    print(f"{name}: {scores.mean():.4f} (+/- {scores.std():.4f})")

LogisticRegression: 0.7935 (+/- 0.0210)
DecisionTree: 0.7767 (+/- 0.0349)
RandomForest: 0.8137 (+/- 0.0314)
XGBoost: 0.8171 (+/- 0.0337)


In [9]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

import pandas as pd
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print(importance)

       feature  importance
1          sex    0.269306
5         fare    0.264815
2          age    0.246012
0       pclass    0.076197
6  family_size    0.051687
3        sibsp    0.029636
9   embarked_S    0.023586
4        parch    0.019965
8   embarked_Q    0.009432
7     is_alone    0.009364
